# Nutrition5k — Data Preparation

Builds four split-aware archives on Google Drive from the official GCS dataset:

| Archive | Content | Used by |
|---------|---------|--------|
| `rgb_train.tar.zst` | side_angles frames, 4059 dishes, pre-resized 292px | Exp1/2 train+val |
| `rgb_test.tar.zst` | side_angles frames, 709 dishes | Exp1/2 eval |
| `depth_train.tar.zst` | realsense_overhead rgb+depth, 2758 dishes | Exp3/4 train+val |
| `depth_test.tar.zst` | realsense_overhead, 507 dishes | Exp3/4 eval |

**Runtime**: A100 Colab, ~2-4 hours for full run (rgb archives are large).  
**Alternative**: Prepare archives locally and upload to Drive — much faster if you have gigabit internet.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import subprocess, sys

subprocess.run(['apt-get', 'install', '-qq', 'zstd', 'ffmpeg'], check=True)

if not subprocess.run(['gsutil', 'ls', 'gs://nutrition5k_dataset/'], capture_output=True).returncode == 0:
    raise RuntimeError('gsutil cannot reach GCS bucket — authenticate first.')

print('Environment ready.')

Environment ready.


In [3]:
import os

DRIVE      = '/content/drive/MyDrive/nutrition5k'
GCS_ROOT   = 'gs://nutrition5k_dataset/nutrition5k_dataset'
SPLITS_DIR = f'{DRIVE}/data/splits'
META_DIR   = f'{DRIVE}/data/metadata'
DATA_DIR   = f'{DRIVE}/data'
TMP_RAW    = '/dev/shm/tmp_raw'
TMP_FRAMES = '/dev/shm/tmp_frames'

for d in [SPLITS_DIR, META_DIR, DATA_DIR, TMP_RAW, TMP_FRAMES]:
    os.makedirs(d, exist_ok=True)

print('Paths configured.')

Paths configured.


## Step 1 — Download metadata and official split files

In [ ]:
# Metadata CSVs
!gsutil -m cp -r {GCS_ROOT}/metadata/ {META_DIR}/

# Official split txt files
!gsutil -m cp '{GCS_ROOT}/dish_ids/splits/*.txt' {SPLITS_DIR}/

for fname in ['rgb_train_ids.txt', 'rgb_test_ids.txt', 'depth_train_ids.txt', 'depth_test_ids.txt']:
    path = f'{SPLITS_DIR}/{fname}'
    n = len(open(path).read().split())
    print(f'{fname}: {n} dishes')

## Step 2 — Build depth archives (overhead, smaller — do first)

In [ ]:
import shutil, subprocess, os

def build_overhead_archive(split_txt, archive_name):
    dish_ids = open(split_txt).read().split()
    print(f'Building {archive_name}: {len(dish_ids)} dishes')

    # Write dish list to a file for gsutil -I
    manifest = '/tmp/dish_list.txt'
    with open(manifest, 'w') as f:
        for did in dish_ids:
            f.write(f'{GCS_ROOT}/imagery/realsense_overhead/{did}/rgb.png\n')
            f.write(f'{GCS_ROOT}/imagery/realsense_overhead/{did}/depth_raw.png\n')

    shutil.rmtree(TMP_RAW, ignore_errors=True)
    os.makedirs(TMP_RAW)

    # Download
    subprocess.run(
        f'cat {manifest} | gsutil -m cp -I {TMP_RAW}/',
        shell=True, check=True
    )

    # Flatten: gsutil -I writes nested paths; restructure to dish_id/rgb.png
    # (gsutil preserves GCS path structure under TMP_RAW)
    realsense_src = f'{TMP_RAW}/nutrition5k_dataset/imagery/realsense_overhead'
    frames_dst    = f'{TMP_FRAMES}/realsense_overhead'
    shutil.rmtree(frames_dst, ignore_errors=True)
    shutil.copytree(realsense_src, frames_dst)

    # Pack
    archive_tmp = f'/dev/shm/{archive_name}'
    subprocess.run(
        ['tar', '--zstd', '-1', '-cf', archive_tmp, '-C', TMP_FRAMES, 'realsense_overhead'],
        check=True
    )

    # Copy to Drive
    shutil.copy2(archive_tmp, f'{DATA_DIR}/{archive_name}')
    os.remove(archive_tmp)
    shutil.rmtree(TMP_RAW, ignore_errors=True)
    shutil.rmtree(frames_dst, ignore_errors=True)
    print(f'Done: {DATA_DIR}/{archive_name}')

build_overhead_archive(f'{SPLITS_DIR}/depth_train_ids.txt', 'depth_train.tar.zst')
build_overhead_archive(f'{SPLITS_DIR}/depth_test_ids.txt',  'depth_test.tar.zst')

## Step 3 — Compute volume estimates (Exp4)

In [4]:
if not os.path.isdir('/content/Nutrition5k'):
    subprocess.run(['git', 'clone', 'https://github.com/T0MYYY/Nutrition5k.git',
                    '/content/Nutrition5k', '-b', 'master', '-q'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Nutrition5k', 'pull', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/Nutrition5k', '-q'], check=True)
if '/content/Nutrition5k' not in sys.path:
    sys.path.insert(0, '/content/Nutrition5k')
import importlib; importlib.invalidate_caches()
print('Package ready.')

Package ready.


In [5]:
import csv
from nutrition5k_pkg.volume import compute_volume_batch_depth_only

overhead_ram = '/dev/shm/realsense_overhead'

# Extract depth_train + depth_test to /dev/shm if not already present
for archive in ['depth_train.tar.zst', 'depth_test.tar.zst']:
    n_present = sum(1 for d in os.scandir(overhead_ram) if d.is_dir()) if os.path.isdir(overhead_ram) else 0
    if n_present < 100:
        print(f'Extracting {archive} ...')
        subprocess.run(['tar', '--zstd', '-xf', f'{DATA_DIR}/{archive}', '-C', '/dev/shm/'], check=True)

dish_ids = sorted(d for d in os.listdir(overhead_ram) if os.path.isdir(os.path.join(overhead_ram, d)))
print(f'Computing volume for {len(dish_ids)} dishes (train + test)...')

volumes = compute_volume_batch_depth_only(depth_dir=overhead_ram, dish_ids=dish_ids)

vol_path = f'{META_DIR}/volume_estimates.csv'
with open(vol_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['dish_id', 'volume_cm3'])
    for did, vol in sorted(volumes.items()):
        w.writerow([did, f'{vol:.4f}'])

print(f'Saved {len(volumes)} volume estimates → {vol_path}')


Extracting depth_train to /dev/shm...
Computing volume for 2755 dishes...
  500/2755 dishes processed
  1000/2755 dishes processed
  1500/2755 dishes processed
  2000/2755 dishes processed
  2500/2755 dishes processed
  2755/2755 dishes processed
Saved 2754 volume estimates → /content/drive/MyDrive/nutrition5k/data/metadata/volume_estimates.csv


## Step 4 — Build RGB archives (side_angles)

**Prepared locally** using `scripts/prepare_data_local.py` (local CPU is faster than Colab CPU for multi-process ffmpeg).

Key difference from naive extraction: instead of all ~130 frames per camera, we keep **6 evenly-spaced frames** per camera per dish.
The dish is a static food plate recorded over ~5 seconds — frames are highly correlated, so 6 frames captures the full information with ~20× less compute.

| | All frames | 6 frames/camera |
|---|---|---|
| Samples (train) | ~2.1 M | ~97 K |
| Epoch time (A100) | ~10 min | ~1.5 min |
| Information loss | baseline | negligible (static dish) |

The code below mirrors the logic in `scripts/prepare_data_local.py` for reference.
To re-build archives on Colab, set `RUN_LOCALLY = False` and re-run.

In [ ]:
# Set True to run on Colab (downloads ~178 GB from GCS).
# Archives were already prepared locally with scripts/prepare_data_local.py.
RUN_ON_COLAB = False

import os, shutil, subprocess, time
from concurrent.futures import ProcessPoolExecutor, as_completed

CAMERAS  = ('A', 'B', 'C', 'D')
N_FRAMES = 6       # evenly-spaced frames to keep per camera per dish
BATCH_SIZE = 200   # dishes per batch; ~8 GB raw video in /dev/shm at a time


def even_indices(n_total, n_select=N_FRAMES):
    """Return n_select evenly-spaced indices spanning [0, n_total-1]."""
    if n_total <= n_select:
        return list(range(n_total))
    return [round(i * (n_total - 1) / (n_select - 1)) for i in range(n_select)]


def extract_dish_frames(args):
    """Extract N_FRAMES evenly-spaced frames per camera, resized to 292px."""
    dish_id, raw_root, frames_root = args
    out_dir = os.path.join(frames_root, dish_id, 'frames')
    os.makedirs(out_dir, exist_ok=True)

    for cam in CAMERAS:
        video = os.path.join(raw_root, dish_id, f'camera_{cam}.h264')
        if not os.path.isfile(video):
            continue
        import tempfile
        with tempfile.TemporaryDirectory() as tmpd:
            # Extract all frames at training resolution
            subprocess.run([
                'ffmpeg', '-i', video,
                '-vf', 'scale=-2:292',
                '-q:v', '3',
                os.path.join(tmpd, 'frame_%06d.jpeg')
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            all_frames = sorted(f for f in os.listdir(tmpd) if f.endswith('.jpeg'))
            if not all_frames:
                continue

            # Keep only N_FRAMES evenly spaced
            selected = [all_frames[i] for i in even_indices(len(all_frames))]
            for j, fname in enumerate(selected, 1):
                src = os.path.join(tmpd, fname)
                dst = os.path.join(out_dir, f'camera_{cam}_frame_{j:04d}.jpeg')
                shutil.copy2(src, dst)
    return dish_id


def build_rgb_archive(split_txt, archive_name, n_workers=10):
    dish_ids = open(split_txt).read().split()
    total = len(dish_ids)
    print(f'Building {archive_name}: {total} dishes, {N_FRAMES} frames/camera')

    tar_intermediate = '/dev/shm/rgb_intermediate.tar'
    archive_tmp      = f'/dev/shm/{archive_name}'
    archive_drive    = f'{DATA_DIR}/{archive_name}'

    for batch_start in range(0, total, BATCH_SIZE):
        batch = dish_ids[batch_start:batch_start + BATCH_SIZE]
        t0 = time.time()

        shutil.rmtree(TMP_RAW, ignore_errors=True)
        os.makedirs(TMP_RAW)
        with open('/tmp/batch_paths.txt', 'w') as f:
            f.write('\n'.join(
                f'{GCS_ROOT}/imagery/side_angles/{did}/' for did in batch
            ))
        subprocess.run(
            f'cat /tmp/batch_paths.txt | gsutil -m cp -r -I {TMP_RAW}/',
            shell=True, check=True
        )

        raw_side = f'{TMP_RAW}/nutrition5k_dataset/imagery/side_angles'
        shutil.rmtree(TMP_FRAMES, ignore_errors=True)
        os.makedirs(f'{TMP_FRAMES}/side_angles')
        args_list = [(did, raw_side, f'{TMP_FRAMES}/side_angles') for did in batch]
        with ProcessPoolExecutor(max_workers=n_workers) as pool:
            for fut in as_completed(pool.submit(extract_dish_frames, a) for a in args_list):
                fut.result()

        subprocess.run(
            ['tar', '-rf', tar_intermediate, '-C', TMP_FRAMES, 'side_angles'],
            check=True
        )
        shutil.rmtree(TMP_RAW, ignore_errors=True)
        shutil.rmtree(TMP_FRAMES, ignore_errors=True)
        print(f'  Batch {batch_start+len(batch)}/{total} done in {time.time()-t0:.0f}s')

    subprocess.run(
        f'zstd -1 -T0 {tar_intermediate} -o {archive_tmp}',
        shell=True, check=True
    )
    os.remove(tar_intermediate)
    shutil.copy2(archive_tmp, archive_drive)
    os.remove(archive_tmp)
    print(f'Done: {archive_drive}')


if RUN_ON_COLAB:
    build_rgb_archive(f'{SPLITS_DIR}/rgb_test_ids.txt',  'rgb_test.tar.zst')
    build_rgb_archive(f'{SPLITS_DIR}/rgb_train_ids.txt', 'rgb_train.tar.zst')
else:
    print('Skipped (RUN_ON_COLAB=False). Archives prepared locally via scripts/prepare_data_local.py')


## Step 5 — Verify archives

In [ ]:
import os

archives = ['rgb_train.tar.zst', 'rgb_test.tar.zst', 'depth_train.tar.zst', 'depth_test.tar.zst']
for a in archives:
    path = f'{DATA_DIR}/{a}'
    if os.path.isfile(path):
        size_gb = os.path.getsize(path) / 1e9
        print(f'  {a}: {size_gb:.1f} GB  ✓')
    else:
        print(f'  {a}: MISSING  ✗')

# Spot-check: list top-level dirs inside rgb_test
import subprocess
result = subprocess.run(
    ['tar', '--zstd', '-tf', f'{DATA_DIR}/rgb_test.tar.zst', '--wildcards', 'side_angles/dish_*'],
    capture_output=True, text=True
)
dishes = sorted(set(p.split('/')[1] for p in result.stdout.split() if p.startswith('side_angles/dish_')))
print(f'rgb_test dishes in archive: {len(dishes)} (expected 709)')